# LangSmith Masterclass

## The 4 important configuration values

1.LANGSMITH_API_KEY - Connects the application to langsmith

2.LANGSMITH_TRACING - Enables the langsmith tracing

3.LANGSMITH_PROJECT - Defines the langsmith project where traces are stored.

4.OPENAI_API_KEY    - Allows the application to use the OpenAI model.

In [5]:
import os

#Langsmith + Open AI Configuration

os.environ["LANGSMITH_API_KEY"] = "xyz"
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] ="PROITBRIDGE_1"
os.environ["OPENAI_API_KEY"]    = "xyzivur"
print("Configuration Set.")

Configuration Set.


# Install required Packages

In [ ]:
%pip install -U langchain langchain-openai langchain-community langchain-text-splitters langsmith pypdf

# Imports

In [4]:
from langsmith import traceable
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters  import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# Part 1 - Simple Python Function Trace

In [8]:
# First Langsmith trace

# @traceable tells Langsmith to record the execution of this function.

@traceable
def calculate_salary(basic_salary,bonus):
    #Simple HR Calculation
    return basic_salary + bonus

result = calculate_salary(50000,10000)
print("Total Salary:",result)

Total Salary: 60000


# Part 2 - RAG Tracing

In [9]:
loader = PyPDFLoader("PROITBRIDGE_Employee_Handbook_2026.pdf")

pages = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=150)
chunks=splitter.split_documents(pages)

print("Handbook Pages:",len(pages),"| chunks:",len(chunks))

Handbook Pages: 57 | chunks: 142


In [10]:
embeddings   = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = InMemoryVectorStore.from_documents(chunks, embeddings)
retriever    = vector_store.as_retriever(search_kwargs={"k": 4})

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

@tool
def search_handbook(query: str) -> str:
    """Search the PROITBRIDGE Employee Handbook and return relevant handbook content."""
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant handbook content was found."
    return "\n\n".join(doc.page_content for doc in docs)

rag_agent = create_agent(model=llm,
                         tools=[search_handbook],
                         system_prompt=("You are an HR handbook assistant. Answer using only information "
                                        "returned by the search_handbook tool. If the handbook does not "
                                        "contain the answer, say that the information was not found."))
print("RAG agent ready.")

RAG agent ready.


# Generate the RAG Trace

In [11]:
result = rag_agent.invoke({"messages": [{"role": "user", "content": "How many casual leave (CL) days do employees get per year?"}]})
print("Answer  :", result["messages"][-1].content)

Answer  : Employees get 8 casual leave (CL) days per year. These 8 days are credited upfront on January 1 each year. Casual leave cannot be carried forward and lapses if not used within the year.


In [12]:
question = "Can I take 4 days of casual leave in a row?"
result = rag_agent.invoke(
                          {"messages": [{"role": "user", "content": question}]},
                          config={"run_name": "RAG"}
                          )

print("Question:", question)
print("Answer  :", result["messages"][-1].content)

Question: Can I take 4 days of casual leave in a row?
Answer  : According to the PROITBRIDGE Employee Handbook, casual leave (CL) can be taken for short, unplanned personal needs and has a maximum limit of 3 consecutive days at a time. Therefore, you cannot take 4 days of casual leave in a row.
